In [5]:
import numpy as np
import pandas as pd
import pickle

INPUT = "global-genre_network-2019.csv"

# Load dataset (tab-separated!)
df = pd.read_csv(INPUT, sep="\t")

# Ensure weight is float
df["weight"] = df["weight"].astype(float)

# --- 1. Unique nodes and mapping ---
genres = pd.unique(df[["source", "target"]].values.ravel())
genre_to_idx = {g: i for i, g in enumerate(genres)}
n = len(genres)

# --- 2. Map edges to indices ---
i = df["source"].map(genre_to_idx).to_numpy()
j = df["target"].map(genre_to_idx).to_numpy()
w = df["weight"].to_numpy()

# --- 3. Build undirected adjacency matrix ---
adj_numpy = np.zeros((n, n), dtype=np.float32)

for a, b, val in zip(i, j, w):
    # Keep max weight if duplicates appear
    if a == b:
        adj_numpy[a, a] = max(adj_numpy[a, a], val)  # self-loop
    else:
        adj_numpy[a, b] = max(adj_numpy[a, b], val)
        adj_numpy[b, a] = max(adj_numpy[b, a], val)

# --- 4. Checks ---
is_symmetric = np.allclose(adj_numpy, adj_numpy.T)
density = np.count_nonzero(adj_numpy) / adj_numpy.size
diag_mass = np.trace(adj_numpy)

print("Nodes:", n)
print("Symmetric:", is_symmetric)
print("Density:", density)
print("Diagonal mass:", diag_mass)

# --- 5. Save ---
np.save("my_graph.npy", adj_numpy)
pickle.dump(genres.tolist(), open("matrix_vocab.pkl", "wb"))

print("Saved my_graph.npy and matrix_vocab.pkl")

Nodes: 310
Symmetric: True
Density: 0.06016649323621228
Diagonal mass: 1561.0
Saved my_graph.npy and matrix_vocab.pkl


In [7]:
print(adj_numpy)

[[282. 479.  24. ...   0.   0.   0.]
 [479. 197.  22. ...   0.   0.   0.]
 [ 24.  22.  73. ...   0.   0.   0.]
 ...
 [  0.   0.   0. ...   0.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]]
